# Parallel reduction: all 7 of Harris's kernels on a T4

Companion lab for the cards *Parallel reduction I: divergence and bank conflicts* and *Parallel reduction II: unrolling and cascading* (perf-3-kernels).

**Runtime → Change runtime type → T4 GPU**, then run all cells.

What you will measure: the time to sum 2^22 ints (16.8 MB) with each kernel from Mark Harris's "Optimizing Parallel Reduction in CUDA", 128 threads per block as in the slides, plus a modern warp-shuffle kernel and a library call (CUB, then `torch.sum`). For each: ms, effective bandwidth in GB/s (bytes read ÷ time), % of the T4's 320 GB/s peak, step and cumulative speedup, and a check against the CPU sum.

Not yet run by the author (no NVIDIA GPU). Colab's T4 clocks and availability vary, so your numbers will too.

In [ ]:
!nvidia-smi

## The kernels

Kernels 1 to 3 are exactly the slides' code apart from a bounds check on the load. Three deliberate changes:
- Every load checks `i < n`, so each level of the multi-launch reduction handles any size.
- Kernels 5 to 7 unroll the last warp. The slides do that with a `volatile` pointer and no synchronization, which relied on the 32 threads of a warp running in lock step. Since Volta (2017) threads in a warp are scheduled independently, so that code can silently give wrong sums. Here each step reads, calls `__syncwarp()`, writes, and calls `__syncwarp()` again (the pattern from NVIDIA's "Using CUDA Warp-Level Primitives" blog). The T4 is Turing, after Volta, so this matters.
- Kernel 7 launches enough blocks to fill the GPU (SMs × 1024 / 128 = 320 on a T4), not the 64 the slides used on the 16-SM G80.

In [ ]:
%%writefile reduction.cu
// Harris, "Optimizing Parallel Reduction in CUDA": kernels 1-7, plus a
// modern warp-shuffle kernel and CUB, on 2^22 ints with 128 threads per block.
// Changes from the slides (all small, all listed):
//  - every load checks i < n, so any n works (the slides assume n fits exactly);
//  - kernels 5-7 replace the "volatile, no sync" last warp with a version that
//    is safe on Volta and later GPUs (read, __syncwarp, write, __syncwarp);
//  - kernel 7 launches enough blocks to fill the GPU, not the slides' 64.
#include <cstdio>
#include <cstdlib>
#include <vector>
#include <cub/cub.cuh>

#define CUDA_CHECK(call) do { cudaError_t e_ = (call); if (e_ != cudaSuccess) { \
  fprintf(stderr, "CUDA error %s at %s:%d\n", cudaGetErrorString(e_), __FILE__, __LINE__); \
  exit(1); } } while (0)

const unsigned int THREADS = 128;

// Kernel 1: interleaved addressing, divergent branch (tid % (2*s))
__global__ void reduce1(const int *g_idata, int *g_odata, unsigned int n) {
  extern __shared__ int sdata[];
  unsigned int tid = threadIdx.x;
  unsigned int i = blockIdx.x * blockDim.x + threadIdx.x;
  sdata[tid] = (i < n) ? g_idata[i] : 0;
  __syncthreads();
  for (unsigned int s = 1; s < blockDim.x; s *= 2) {
    if (tid % (2 * s) == 0) sdata[tid] += sdata[tid + s];
    __syncthreads();
  }
  if (tid == 0) g_odata[blockIdx.x] = sdata[0];
}

// Kernel 2: interleaved addressing, strided index (bank conflicts)
__global__ void reduce2(const int *g_idata, int *g_odata, unsigned int n) {
  extern __shared__ int sdata[];
  unsigned int tid = threadIdx.x;
  unsigned int i = blockIdx.x * blockDim.x + threadIdx.x;
  sdata[tid] = (i < n) ? g_idata[i] : 0;
  __syncthreads();
  for (unsigned int s = 1; s < blockDim.x; s *= 2) {
    unsigned int index = 2 * s * tid;
    if (index < blockDim.x) sdata[index] += sdata[index + s];
    __syncthreads();
  }
  if (tid == 0) g_odata[blockIdx.x] = sdata[0];
}

// Kernel 3: sequential addressing
__global__ void reduce3(const int *g_idata, int *g_odata, unsigned int n) {
  extern __shared__ int sdata[];
  unsigned int tid = threadIdx.x;
  unsigned int i = blockIdx.x * blockDim.x + threadIdx.x;
  sdata[tid] = (i < n) ? g_idata[i] : 0;
  __syncthreads();
  for (unsigned int s = blockDim.x / 2; s > 0; s >>= 1) {
    if (tid < s) sdata[tid] += sdata[tid + s];
    __syncthreads();
  }
  if (tid == 0) g_odata[blockIdx.x] = sdata[0];
}

// Kernel 4: first add during global load (each block covers 2*blockDim elements)
__global__ void reduce4(const int *g_idata, int *g_odata, unsigned int n) {
  extern __shared__ int sdata[];
  unsigned int tid = threadIdx.x;
  unsigned int i = blockIdx.x * (blockDim.x * 2) + threadIdx.x;
  int v = (i < n) ? g_idata[i] : 0;
  if (i + blockDim.x < n) v += g_idata[i + blockDim.x];
  sdata[tid] = v;
  __syncthreads();
  for (unsigned int s = blockDim.x / 2; s > 0; s >>= 1) {
    if (tid < s) sdata[tid] += sdata[tid + s];
    __syncthreads();
  }
  if (tid == 0) g_odata[blockIdx.x] = sdata[0];
}

// Last-warp reduction. The slides use "volatile" and no sync, which relied on
// lock-step warps; Volta's independent thread scheduling breaks that. Here each
// step reads, syncs the warp, writes, syncs again (NVIDIA's warp-primitives blog).
// Needs blockSize >= 64 (it reads sdata[tid + 32]).
template <unsigned int blockSize>
__device__ void warpReduce(volatile int *sdata, unsigned int tid) {
  int v = sdata[tid];
  if (blockSize >= 64) { v += sdata[tid + 32]; __syncwarp(); sdata[tid] = v; __syncwarp(); }
  if (blockSize >= 32) { v += sdata[tid + 16]; __syncwarp(); sdata[tid] = v; __syncwarp(); }
  if (blockSize >= 16) { v += sdata[tid + 8];  __syncwarp(); sdata[tid] = v; __syncwarp(); }
  if (blockSize >= 8)  { v += sdata[tid + 4];  __syncwarp(); sdata[tid] = v; __syncwarp(); }
  if (blockSize >= 4)  { v += sdata[tid + 2];  __syncwarp(); sdata[tid] = v; __syncwarp(); }
  if (blockSize >= 2)  { v += sdata[tid + 1];  __syncwarp(); sdata[tid] = v; __syncwarp(); }
}

// Kernel 5: unroll the last warp (loop stops at s > 32)
__global__ void reduce5(const int *g_idata, int *g_odata, unsigned int n) {
  extern __shared__ int sdata[];
  unsigned int tid = threadIdx.x;
  unsigned int i = blockIdx.x * (blockDim.x * 2) + threadIdx.x;
  int v = (i < n) ? g_idata[i] : 0;
  if (i + blockDim.x < n) v += g_idata[i + blockDim.x];
  sdata[tid] = v;
  __syncthreads();
  for (unsigned int s = blockDim.x / 2; s > 32; s >>= 1) {
    if (tid < s) sdata[tid] += sdata[tid + s];
    __syncthreads();
  }
  if (tid < 32) warpReduce<64>(sdata, tid);
  if (tid == 0) g_odata[blockIdx.x] = sdata[0];
}

// Kernel 6: completely unrolled with a template block size
template <unsigned int blockSize>
__global__ void reduce6(const int *g_idata, int *g_odata, unsigned int n) {
  extern __shared__ int sdata[];
  unsigned int tid = threadIdx.x;
  unsigned int i = blockIdx.x * (blockSize * 2) + threadIdx.x;
  int v = (i < n) ? g_idata[i] : 0;
  if (i + blockSize < n) v += g_idata[i + blockSize];
  sdata[tid] = v;
  __syncthreads();
  if (blockSize >= 512) { if (tid < 256) sdata[tid] += sdata[tid + 256]; __syncthreads(); }
  if (blockSize >= 256) { if (tid < 128) sdata[tid] += sdata[tid + 128]; __syncthreads(); }
  if (blockSize >= 128) { if (tid < 64)  sdata[tid] += sdata[tid + 64];  __syncthreads(); }
  if (tid < 32) warpReduce<blockSize>(sdata, tid);
  if (tid == 0) g_odata[blockIdx.x] = sdata[0];
}

// Kernel 7: multiple elements per thread (grid-stride loop keeps loads coalesced)
template <unsigned int blockSize>
__global__ void reduce7(const int *g_idata, int *g_odata, unsigned int n) {
  extern __shared__ int sdata[];
  unsigned int tid = threadIdx.x;
  unsigned int i = blockIdx.x * (blockSize * 2) + threadIdx.x;
  unsigned int gridSize = blockSize * 2 * gridDim.x;
  int v = 0;
  while (i < n) {
    v += g_idata[i];
    if (i + blockSize < n) v += g_idata[i + blockSize];
    i += gridSize;
  }
  sdata[tid] = v;
  __syncthreads();
  if (blockSize >= 512) { if (tid < 256) sdata[tid] += sdata[tid + 256]; __syncthreads(); }
  if (blockSize >= 256) { if (tid < 128) sdata[tid] += sdata[tid + 128]; __syncthreads(); }
  if (blockSize >= 128) { if (tid < 64)  sdata[tid] += sdata[tid + 64];  __syncthreads(); }
  if (tid < 32) warpReduce<blockSize>(sdata, tid);
  if (tid == 0) g_odata[blockIdx.x] = sdata[0];
}

// Modern kernel (not in the slides): kernel 7's loads, then warp shuffles.
// No shared-memory tree, only one shared int per warp.
__device__ __forceinline__ int warpSum(int v) {
  for (int offset = 16; offset > 0; offset >>= 1)
    v += __shfl_down_sync(0xffffffffu, v, offset);
  return v;
}

template <unsigned int blockSize>
__global__ void reduceShfl(const int *g_idata, int *g_odata, unsigned int n) {
  __shared__ int warpSums[blockSize / 32];
  unsigned int tid = threadIdx.x;
  unsigned int i = blockIdx.x * (blockSize * 2) + threadIdx.x;
  unsigned int gridSize = blockSize * 2 * gridDim.x;
  int v = 0;
  while (i < n) {
    v += g_idata[i];
    if (i + blockSize < n) v += g_idata[i + blockSize];
    i += gridSize;
  }
  v = warpSum(v);
  if ((tid & 31) == 0) warpSums[tid >> 5] = v;
  __syncthreads();
  if (tid < 32) {
    v = (tid < blockSize / 32) ? warpSums[tid] : 0;
    v = warpSum(v);
    if (tid == 0) g_odata[blockIdx.x] = v;
  }
}

// ---- host side ----
typedef void (*KernelFn)(const int *, int *, unsigned int);

struct Variant {
  const char *name;
  KernelFn fn;
  unsigned int elemsPerBlock;  // elements one block covers per pass (ignoring grid-stride)
  bool capped;                 // grid-stride kernels: cap the number of blocks
};

unsigned int g_maxBlocks = 64;

// Reduce n ints from d_in to one int. Level by level: each launch is the global sync.
// Returns a device pointer to the final value (in bufA or bufB).
int *runReduction(const Variant &v, const int *d_in, int *bufA, int *bufB, unsigned int n) {
  const int *src = d_in;
  int *dst = bufA;
  size_t smem = THREADS * sizeof(int);
  while (true) {
    unsigned int blocks = (n + v.elemsPerBlock - 1) / v.elemsPerBlock;
    if (v.capped && blocks > g_maxBlocks) blocks = g_maxBlocks;
    v.fn<<<blocks, THREADS, smem>>>(src, dst, n);
    CUDA_CHECK(cudaGetLastError());
    if (blocks == 1) return dst;
    n = blocks;
    src = dst;
    dst = (dst == bufA) ? bufB : bufA;
  }
}

int main() {
  const unsigned int N = 1u << 22;          // 4M ints, as in the slides
  const size_t bytes = (size_t)N * sizeof(int);
  const double T4_PEAK_GBS = 320.0;
  const int WARMUP = 5, REPS = 100;

  cudaDeviceProp prop;
  CUDA_CHECK(cudaGetDeviceProperties(&prop, 0));
  g_maxBlocks = prop.multiProcessorCount * (prop.maxThreadsPerMultiProcessor / THREADS);
  printf("GPU: %s, %d SMs, kernel 7 / shuffle use %u blocks (slides: 64 on G80)\n",
         prop.name, prop.multiProcessorCount, g_maxBlocks);

  std::vector<int> h(N);
  srand(0);
  long long cpu = 0;
  for (unsigned int k = 0; k < N; ++k) { h[k] = rand() & 0x3F; cpu += h[k]; }

  int *d_in, *bufA, *bufB;
  CUDA_CHECK(cudaMalloc(&d_in, bytes));
  CUDA_CHECK(cudaMalloc(&bufA, (N / THREADS + 1) * sizeof(int)));
  CUDA_CHECK(cudaMalloc(&bufB, (N / THREADS + 1) * sizeof(int)));
  CUDA_CHECK(cudaMemcpy(d_in, h.data(), bytes, cudaMemcpyHostToDevice));

  const Variant variants[] = {
    {"1 interleaved, divergent branch", reduce1,              THREADS,     false},
    {"2 interleaved, bank conflicts",   reduce2,              THREADS,     false},
    {"3 sequential addressing",         reduce3,              THREADS,     false},
    {"4 first add during load",         reduce4,              THREADS * 2, false},
    {"5 unroll last warp",              reduce5,              THREADS * 2, false},
    {"6 completely unrolled",           reduce6<THREADS>,     THREADS * 2, false},
    {"7 multiple elements per thread",  reduce7<THREADS>,     THREADS * 2, true},
    {"+ warp shuffle (modern)",         reduceShfl<THREADS>,  THREADS * 2, true},
  };
  const int NV = sizeof(variants) / sizeof(variants[0]);

  cudaEvent_t start, stop;
  CUDA_CHECK(cudaEventCreate(&start));
  CUDA_CHECK(cudaEventCreate(&stop));

  printf("\n%-34s %9s %9s %8s %7s %7s  %s\n", "kernel", "ms", "GB/s", "% of 320", "step", "cumul", "check");
  double first = 0, prev = 0;
  for (int k = 0; k < NV; ++k) {
    int *d_res = nullptr;
    for (int w = 0; w < WARMUP; ++w) d_res = runReduction(variants[k], d_in, bufA, bufB, N);
    CUDA_CHECK(cudaDeviceSynchronize());
    CUDA_CHECK(cudaEventRecord(start));
    for (int r = 0; r < REPS; ++r) d_res = runReduction(variants[k], d_in, bufA, bufB, N);
    CUDA_CHECK(cudaEventRecord(stop));
    CUDA_CHECK(cudaEventSynchronize(stop));
    float total;
    CUDA_CHECK(cudaEventElapsedTime(&total, start, stop));
    double ms = total / REPS;
    int gpu;
    CUDA_CHECK(cudaMemcpy(&gpu, d_res, sizeof(int), cudaMemcpyDeviceToHost));
    double gbs = bytes / (ms * 1e-3) / 1e9;
    if (k == 0) first = ms;
    char step[16] = "-";
    if (k > 0) snprintf(step, sizeof(step), "%.2fx", prev / ms);
    printf("%-34s %9.4f %9.2f %7.1f%% %7s %6.2fx  %s\n", variants[k].name, ms, gbs,
           100.0 * gbs / T4_PEAK_GBS, step, first / ms, (gpu == cpu) ? "ok" : "WRONG");
    prev = ms;
  }

  // Library call: CUB DeviceReduce::Sum (two-phase: query temp size, then run)
  int *d_out;
  void *d_temp = nullptr;
  size_t tempBytes = 0;
  CUDA_CHECK(cudaMalloc(&d_out, sizeof(int)));
  CUDA_CHECK(cub::DeviceReduce::Sum(d_temp, tempBytes, d_in, d_out, (int)N));
  CUDA_CHECK(cudaMalloc(&d_temp, tempBytes));
  for (int w = 0; w < WARMUP; ++w) CUDA_CHECK(cub::DeviceReduce::Sum(d_temp, tempBytes, d_in, d_out, (int)N));
  CUDA_CHECK(cudaDeviceSynchronize());
  CUDA_CHECK(cudaEventRecord(start));
  for (int r = 0; r < REPS; ++r) CUDA_CHECK(cub::DeviceReduce::Sum(d_temp, tempBytes, d_in, d_out, (int)N));
  CUDA_CHECK(cudaEventRecord(stop));
  CUDA_CHECK(cudaEventSynchronize(stop));
  float total;
  CUDA_CHECK(cudaEventElapsedTime(&total, start, stop));
  double ms = total / REPS;
  int gpu;
  CUDA_CHECK(cudaMemcpy(&gpu, d_out, sizeof(int), cudaMemcpyDeviceToHost));
  double gbs = bytes / (ms * 1e-3) / 1e9;
  printf("%-34s %9.4f %9.2f %7.1f%% %7s %6.2fx  %s\n", "library: cub::DeviceReduce::Sum", ms, gbs,
         100.0 * gbs / T4_PEAK_GBS, "-", first / ms, (gpu == cpu) ? "ok" : "WRONG");
  printf("\nCPU sum = %lld (ints in 0..63, so even 2^25 of them fit in an int)\n", cpu);

  CUDA_CHECK(cudaFree(d_temp)); CUDA_CHECK(cudaFree(d_out));
  CUDA_CHECK(cudaFree(d_in)); CUDA_CHECK(cudaFree(bufA)); CUDA_CHECK(cudaFree(bufB));
  CUDA_CHECK(cudaEventDestroy(start)); CUDA_CHECK(cudaEventDestroy(stop));
  return 0;
}

In [ ]:
!nvcc -O3 -arch=sm_75 -o reduction reduction.cu && ./reduction

## The same sum with PyTorch

`torch.sum` calls a library reduction kernel. Same 2^22 int32 values, timed with CUDA events after warm-up.

In [ ]:
import torch
N = 1 << 22
x = torch.randint(0, 64, (N,), dtype=torch.int32, device="cuda")
for _ in range(5):
    y = x.sum()
torch.cuda.synchronize()
start, stop = torch.cuda.Event(enable_timing=True), torch.cuda.Event(enable_timing=True)
reps = 100
start.record()
for _ in range(reps):
    y = x.sum()
stop.record()
torch.cuda.synchronize()
ms = start.elapsed_time(stop) / reps
gbs = N * 4 / (ms * 1e-3) / 1e9
print(f"torch.sum: {ms:.4f} ms, {gbs:.1f} GB/s, {100 * gbs / 320:.1f}% of 320 GB/s")
print("check:", int(y.item()) == int(x.long().sum().item()))

## Compare with the slides (NVIDIA G80, 86.4 GB/s peak, 2^22 ints, 128 threads per block)

| Kernel | Time | Bandwidth | % of 86.4 GB/s | Step | Cumulative |
|---|---|---|---|---|---|
| 1 interleaved addressing, divergent branching | 8.054 ms | 2.083 GB/s | 2.4% | | |
| 2 interleaved addressing, bank conflicts | 3.456 ms | 4.854 GB/s | 5.6% | 2.33× | 2.33× |
| 3 sequential addressing | 1.722 ms | 9.741 GB/s | 11.3% | 2.01× | 4.68× |
| 4 first add during global load | 0.965 ms | 17.377 GB/s | 20.1% | 1.78× | 8.34× |
| 5 unroll last warp | 0.536 ms | 31.289 GB/s | 36.2% | 1.8× | 15.01× |
| 6 completely unrolled | 0.381 ms | 43.996 GB/s | 50.9% | 1.41× | 21.16× |
| 7 multiple elements per thread | 0.268 ms | 62.671 GB/s | 72.5% | 1.42× | 30.04× |

Time, bandwidth and speedups are from the slides; the % column is our recomputation (bandwidth ÷ 86.4). Kernel 7 on 32M elements reached 73 GB/s.

Your T4 ladder will differ (our guess, not a measurement): the T4 is a much newer design, and the whole 16.8 MB array needs only 52 µs at 320 GB/s, so launch overhead (4 launches for kernels 1 to 3) is a large share of the time.

## Try this
1. Change `THREADS` to 256 or 512 (and the template arguments follow). Does kernel 1 get worse or better?
2. Set `g_maxBlocks = 64` in `main` to copy the slides' kernel 7 launch. How far does bandwidth drop on 40 SMs?
3. Change `N` to `1u << 25` (32M ints, as in the slides' 73 GB/s note). Which kernels get closer to 320 GB/s once launch overhead is amortized?